In [22]:
print('Đang khai báo thư viện')

import joblib
import pandas as pd
import os
from sklearn.metrics import accuracy_score, f1_score, classification_report

print('Khai báo thư viện thành công')

Đang khai báo thư viện
Khai báo thư viện thành công


In [23]:
model_path   = os.path.join("..", "Model", "xgboost_final_model.pkl")
feature_path = os.path.join("..", "Model", "feature_names_xgb_final.pkl")

model         = joblib.load(model_path)
feature_names = joblib.load(feature_path)
print(f"Đã load final model")
print(f"Features ({len(feature_names)}): {feature_names}")

Đã load final model
Features (16): ['Popularity', 'danceability', 'energy', 'key', 'loudness', 'mode', 'speechiness', 'acousticness', 'instrumentalness', 'liveness', 'valence', 'tempo', 'duration_ms', 'time_signature', 'key_sin', 'key_cos']


In [24]:
# DỰ ĐOÁN TRÊN TẬP TEST

# Load test data
test_path = os.path.join("..", "Data", "DataCleaned", "test_cleaned.csv")
df_test   = pd.read_csv(test_path)
print(f"\nTest shape: {df_test.shape}")

# Drop các cột không cần thiết
drop_cols          = ['Id', 'Artist Name', 'Track Name']
existing_drop_test = [col for col in drop_cols if col in df_test.columns]

if 'Class' in df_test.columns:
    X_test    = df_test.drop(columns=['Class'] + existing_drop_test)
    y_test    = df_test['Class']
    has_label = True
else:
    X_test    = df_test.drop(columns=existing_drop_test)
    has_label = False

X_test = X_test[feature_names]
print(f"X_test shape sau khi align features: {X_test.shape}")


Test shape: (3600, 16)
X_test shape sau khi align features: (3600, 16)


In [25]:
y_pred_test = model.predict(X_test)

print("\n=== KẾT QUẢ DỰ ĐOÁN TRÊN TEST ===")
print(f"Số lượng mẫu test: {len(y_pred_test)}")
print(f"Phân bố class dự đoán:\n{pd.Series(y_pred_test).value_counts().sort_index()}")

if has_label:
    print(f"\nAccuracy Test : {accuracy_score(y_test, y_pred_test):.4f}")
    print(f"Macro F1 Test : {f1_score(y_test, y_pred_test, average='macro', zero_division=0):.4f}")
    print("\nClassification Report:")
    print(classification_report(y_test, y_pred_test, zero_division=0))

# Xuất submission
submission = pd.DataFrame({
    'Id'   : df_test['Id'] if 'Id' in df_test.columns else range(len(y_pred_test)),
    'Class': y_pred_test
})
submission.to_csv('submission_xgb_final.csv', index=False)
print("\nĐã lưu kết quả vào submission_xgb_final.csv")


=== KẾT QUẢ DỰ ĐOÁN TRÊN TEST ===
Số lượng mẫu test: 3600
Phân bố class dự đoán:
0     206
1     296
2     340
3      91
4     112
5     302
6     439
7     122
8     465
9     523
10    704
Name: count, dtype: int64

Đã lưu kết quả vào submission_xgb_final.csv


In [26]:
sample_path = os.path.join("..", "Data", "sample_submission.csv")
sample = pd.read_csv(sample_path)

print(f"Sample shape: {sample.shape}")
print(f"Sample Id range: {sample['Id'].min()} - {sample['Id'].max()}")
print(f"First 10 Ids: {sample['Id'].head(10).tolist()}")

# Lấy đúng Id từ sample
correct_ids = sample['Id'].values

print(f"Number of predictions: {len(y_pred_test)}")

# Kiểm tra số lượng khớp nhau
if len(correct_ids) != len(y_pred_test):
    print(f"Warning: Sample IDs ({len(correct_ids)}) != Predictions ({len(y_pred_test)})")
    min_len = min(len(correct_ids), len(y_pred_test))
    correct_ids = correct_ids[:min_len]
    y_pred_adjusted = y_pred_test[:min_len]
else:
    y_pred_adjusted = y_pred_test

# Tạo submission với Id đúng
submission_fixed = pd.DataFrame({
    'Id': correct_ids,
    'Class': y_pred_adjusted
})

output_dir = os.path.join("..", "Submissions") 
os.makedirs(output_dir, exist_ok=True)     

output_path = os.path.join(output_dir, "submission_xgb.csv")
submission_fixed.to_csv(output_path, index=False)

print(f"\nSaved: {output_path}")
print(f"Submission shape: {submission_fixed.shape}")
print(f"Id range: {submission_fixed['Id'].min()} - {submission_fixed['Id'].max()}")

# Kiểm tra nhanh phân bố class trong submission
print(f"\nClass distribution in submission:")
print(submission_fixed['Class'].value_counts().sort_index())

print("\nFile ready for Kaggle submission!")

Sample shape: (3600, 2)
Sample Id range: 14397 - 17996
First 10 Ids: [14397, 14398, 14399, 14400, 14401, 14402, 14403, 14404, 14405, 14406]
Number of predictions: 3600

Saved: ..\Submissions\submission_xgb.csv
Submission shape: (3600, 2)
Id range: 14397 - 17996

Class distribution in submission:
Class
0     206
1     296
2     340
3      91
4     112
5     302
6     439
7     122
8     465
9     523
10    704
Name: count, dtype: int64

File ready for Kaggle submission!
